In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# handeling the missing values
df = df.fillna(df.mean())
check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols)) # cheack first do we have categorical columns?
if categorical_cols.empty == True :
  print("there is no categorical columns")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 5: Write your code here:
# Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")
balance = df['Target'].value_counts(normalize=True)
if balance[0] > 0.5 or balance[1] > 0.5 :
  print("there is imbalance")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)


In [ ]:
%pip install kagglehub catboost xgboost tqdm -q

from catboost import CatBoostClassifier

In [ ]:
# Task 2,3,4,5: Write your code here:

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42) # Stratified KFold because there is imbalance

model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4)

from sklearn.metrics import accuracy_score, f1_score
F1_scores = []
accuracy_scores = []
# training loop
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)
  # Calculate metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)
  # store results
  F1_scores.append(f1)
  accuracy_scores.append(accuracy)

print(f"  F1 Average scores:  {np.mean(F1_scores):.4f}")
print(f"  accuracy Average scores:  {np.mean(accuracy_scores):.4f}")


In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))  # because it's a lot of features we will identify the top 10 important features
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(1)
print("the moost important feature is: ", importance)

In [ ]:
# Task Bonus: Write your code here:
X_gold = df['P_2'].astype(float) # ONLY THE GOLDEN FEATURE

F1Golden_scores = []  # for the golden feature only
accuracyGolden_scores = [] # for the golden feature only
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)): # RUNNING K FOLD AGIN
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)
  # Calculate metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)
  # store results
  F1Golden_scores.append(f1)
  accuracyGolden_scores.append(accuracy)

print(f"  F1 Golden feature Average scores:  {np.mean(F1Golden_scores):.4f}")
print(f"  accuracy Golden feature Average scores:  {np.mean(accuracyGolden_scores):.4f}")

print(f"  F1 for full model:  {np.mean(F1_scores):.4f}")
print(f"  accuracy for full model:  {np.mean(accuracy_scores):.4f}")

print(f"accuracy for golden feature only: {np.mean(accuracyGolden_scores):.4f}")
print(f"  F1 for golden feature only:  {np.mean(F1Golden_scores):.4f}")




